# Module 3: Vector Databases for RAG
## Topics Covered
- Vector Databases: Architecture & Concepts
- Embedding Storage & Semantic Search
- ChromaDB: Architecture, Vector Operations, Similarity Search
- Embedding Models
- Recommendation Systems

---
**Real-world scenario**: Building a semantic search engine and recommendation system for a movie streaming platform.

In [ ]:
%pip install -q chromadb langchain-openai sentence-transformers numpy pandas scikit-learn

In [ ]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("Ready:", OPENAI_API_KEY != "your-api-key-here")

---
## 1. Understanding Vectors and Embeddings

**Embeddings** map text to high-dimensional vectors that capture semantic meaning.

```
"king" → [0.2, -0.5, 0.8, ...] (1536 dimensions with OpenAI)
"queen" → [0.25, -0.48, 0.79, ...] (similar direction!)
```

**Key insight**: Similar meanings → similar vectors → small angular distance

In [ ]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

# Initialize embeddings model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Demonstrate semantic similarity
texts = [
    "The movie has incredible action sequences",       # Action
    "Explosive fight scenes throughout the film",      # Similar to above
    "A heartfelt romantic story about love",           # Romance
    "Beautiful love story that made me cry",           # Similar to above
    "Python programming tutorial for beginners"        # Unrelated
]

vectors = embeddings.embed_documents(texts)
vectors_np = np.array(vectors)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Vector dimensions: {len(vectors[0])}")
print("\nSemantic Similarity Scores (1.0 = identical meaning):")
print(f"Action vs Action-similar: {cosine_similarity(vectors_np[0], vectors_np[1]):.4f} ← HIGH")
print(f"Action vs Romance:        {cosine_similarity(vectors_np[0], vectors_np[2]):.4f} ← MEDIUM")
print(f"Action vs Python tutorial: {cosine_similarity(vectors_np[0], vectors_np[4]):.4f} ← LOW")
print(f"Romance vs Romance-similar: {cosine_similarity(vectors_np[2], vectors_np[3]):.4f} ← HIGH")

---
## 2. ChromaDB Architecture

**ChromaDB** is an open-source vector database designed for AI applications.

```
ChromaDB Architecture:
┌─────────────────────────────────────────────┐
│                 Collections                  │
│  ┌──────────────────────────────────────┐   │
│  │  Document + Embedding + Metadata     │   │
│  │  {id, text, vector[1536], {tags}}    │   │
│  └──────────────────────────────────────┘   │
│  HNSW Index (Hierarchical Navigable          │
│             Small World graph)               │
│  Storage: In-memory or persistent on disk    │
└─────────────────────────────────────────────┘
```

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# In-memory ChromaDB (ephemeral)
client = chromadb.Client()

# Persistent ChromaDB (saves to disk)
# client = chromadb.PersistentClient(path="./chroma_db")

# Create a collection with OpenAI embeddings
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small"
)

# Create or get existing collection
movies_collection = client.get_or_create_collection(
    name="movies",
    embedding_function=openai_ef,
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity
)

print(f"Collection created: {movies_collection.name}")
print(f"Count: {movies_collection.count()}")

In [ ]:
# Movie catalog data
movie_catalog = [
    {"id": "m001", "title": "The Dark Knight", "genre": "Action/Crime", "year": 2008,
     "description": "Batman fights the Joker in Gotham. Dark psychological thriller with explosive action and moral dilemmas."},
    {"id": "m002", "title": "Inception", "genre": "Sci-Fi/Thriller", "year": 2010,
     "description": "A thief enters people's dreams to steal secrets. Mind-bending heist film with stunning visuals."},
    {"id": "m003", "title": "The Notebook", "genre": "Romance/Drama", "year": 2004,
     "description": "A passionate love story between two people from different social backgrounds spanning decades."},
    {"id": "m004", "title": "Avengers: Endgame", "genre": "Action/Superhero", "year": 2019,
     "description": "Superheroes unite to reverse Thanos's catastrophic actions in an epic superhero battle."},
    {"id": "m005", "title": "La La Land", "genre": "Romance/Musical", "year": 2016,
     "description": "A jazz musician and aspiring actress fall in love in Los Angeles pursuing their dreams."},
    {"id": "m006", "title": "Get Out", "genre": "Horror/Thriller", "year": 2017,
     "description": "A Black man visits his white girlfriend's family and uncovers a disturbing secret. Social horror."},
    {"id": "m007", "title": "Parasite", "genre": "Thriller/Drama", "year": 2019,
     "description": "A poor Korean family infiltrates a wealthy household in this Oscar-winning dark comedy thriller."},
    {"id": "m008", "title": "Toy Story", "genre": "Animation/Family", "year": 1995,
     "description": "Toys come to life when humans aren't around. A friendship story between Woody and Buzz."},
    {"id": "m009", "title": "Mad Max: Fury Road", "genre": "Action/Adventure", "year": 2015,
     "description": "In a post-apocalyptic wasteland, survivors race across the desert in a high-octane action spectacle."},
    {"id": "m010", "title": "Pride & Prejudice", "genre": "Romance/Drama", "year": 2005,
     "description": "Elizabeth Bennet navigates love, marriage, and social expectations in 19th century England."}
]

# Add to ChromaDB
movies_collection.add(
    ids=[m["id"] for m in movie_catalog],
    documents=[m["description"] for m in movie_catalog],
    metadatas=[{"title": m["title"], "genre": m["genre"], "year": m["year"]} for m in movie_catalog]
)

print(f"Added {movies_collection.count()} movies to ChromaDB")

---
## 3. Vector Operations & Similarity Search

In [ ]:
# Basic semantic search
def semantic_search(query, n_results=3, genre_filter=None):
    """Search movies by semantic meaning"""
    where = {"genre": {"$contains": genre_filter}} if genre_filter else None
    
    results = movies_collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where,
        include=["documents", "metadatas", "distances"]
    )
    
    print(f"🔍 Query: '{query}'")
    print(f"Top {n_results} results:\n")
    for i, (meta, dist) in enumerate(zip(results['metadatas'][0], results['distances'][0])):
        similarity = 1 - dist  # Convert distance to similarity
        print(f"{i+1}. 🎬 {meta['title']} ({meta['year']})")
        print(f"   Genre: {meta['genre']}")
        print(f"   Similarity: {similarity:.3f}")
    print()

# Test different search queries
semantic_search("exciting battles and superhero fights")
semantic_search("touching romantic love stories")
semantic_search("psychological suspense with twists")

In [ ]:
# Metadata filtering combined with semantic search
# Find action movies similar to a description
results = movies_collection.query(
    query_texts=["epic battles with heroes saving the world"],
    n_results=5,
    where={"year": {"$gte": 2010}},  # Only movies from 2010 onwards
    include=["metadatas", "distances", "documents"]
)

print("Movies from 2010+ about epic battles:\n")
for meta, doc, dist in zip(results['metadatas'][0], results['documents'][0], results['distances'][0]):
    print(f"🎬 {meta['title']} ({meta['year']}) | Genre: {meta['genre']}")
    print(f"   {doc[:80]}...")
    print(f"   Similarity: {(1-dist):.3f}\n")

---
## 4. Embedding Models Comparison

Different embedding models have different trade-offs:

| Model | Dimensions | Speed | Cost | Best For |
|-------|-----------|-------|------|----------|
| text-embedding-3-small | 1536 | Fast | Low | General purpose |
| text-embedding-3-large | 3072 | Medium | Medium | High accuracy |
| all-MiniLM-L6-v2 | 384 | Very Fast | Free | Local/offline |
| BAAI/bge-large-en | 1024 | Medium | Free | Production local |

In [ ]:
# Using sentence-transformers (free, local embeddings)
from sentence_transformers import SentenceTransformer
import numpy as np

# Lightweight local embedding model
local_model = SentenceTransformer('all-MiniLM-L6-v2')

# Compare embedding quality
sentences = [
    "The restaurant serves excellent pasta dishes",
    "Great Italian food with delicious noodles",      # Semantically similar
    "The weather is sunny and warm today",             # Unrelated
]

local_embeddings = local_model.encode(sentences)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Local model dimensions: {local_embeddings.shape[1]}")
print(f"Sentence 1 vs 2 (should be HIGH): {cosine_sim(local_embeddings[0], local_embeddings[1]):.4f}")
print(f"Sentence 1 vs 3 (should be LOW):  {cosine_sim(local_embeddings[0], local_embeddings[2]):.4f}")

# ChromaDB with local sentence-transformers
local_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

local_collection = client.get_or_create_collection(
    name="movies_local_embeddings",
    embedding_function=local_ef
)

# Add same movies with local embeddings
local_collection.add(
    ids=[m["id"] for m in movie_catalog],
    documents=[m["description"] for m in movie_catalog],
    metadatas=[{"title": m["title"], "genre": m["genre"]} for m in movie_catalog]
)
print(f"\nLocal embeddings collection: {local_collection.count()} movies")

---
## 5. Recommendation System with Vector Similarity

In [ ]:
def get_recommendations(movie_title, n_recommendations=3):
    """
    Content-based movie recommendations using vector similarity.
    'Users who liked X will also like Y'
    """
    # Find the movie's embedding (fetch by metadata)
    results = movies_collection.get(
        where={"title": {"$eq": movie_title}},
        include=["documents", "embeddings", "metadatas"]
    )
    
    if not results['ids']:
        return f"Movie '{movie_title}' not found"
    
    movie_id = results['ids'][0]
    movie_doc = results['documents'][0]
    movie_meta = results['metadatas'][0]
    
    # Find similar movies (excluding the movie itself)
    similar = movies_collection.query(
        query_texts=[movie_doc],
        n_results=n_recommendations + 1,
        include=["metadatas", "distances", "documents"]
    )
    
    print(f"🎬 Because you watched: {movie_title}")
    print(f"Genre: {movie_meta['genre']}")
    print(f"\n🍿 You might also like:\n")
    
    shown = 0
    for meta, doc, dist in zip(similar['metadatas'][0], similar['documents'][0], similar['distances'][0]):
        if meta['title'] == movie_title:
            continue
        similarity_pct = (1 - dist) * 100
        print(f"  ✅ {meta['title']} ({meta['year']})")
        print(f"     Genre: {meta['genre']}")
        print(f"     Match: {similarity_pct:.1f}%")
        print(f"     {doc[:80]}...\n")
        shown += 1
        if shown >= n_recommendations:
            break

# Test recommendations
get_recommendations("The Dark Knight")
print("="*60)
get_recommendations("La La Land")

In [ ]:
# Hybrid recommendation: User profile based on watch history
def profile_based_recommendations(watch_history: list, n_results=4):
    """
    Build user taste profile from watch history and recommend new movies.
    Averages embeddings of watched movies = user preference vector.
    """
    # Get embeddings for watched movies
    watched_embeddings = []
    for title in watch_history:
        result = movies_collection.get(
            where={"title": {"$eq": title}},
            include=["embeddings"]
        )
        if result['embeddings']:
            watched_embeddings.append(result['embeddings'][0])
    
    if not watched_embeddings:
        return "No watched movies found"
    
    # Create user profile = average of watched movie embeddings
    user_profile_vector = np.mean(watched_embeddings, axis=0).tolist()
    
    # Find movies similar to user profile
    recommendations = movies_collection.query(
        query_embeddings=[user_profile_vector],
        n_results=n_results + len(watch_history),
        include=["metadatas", "distances"]
    )
    
    print(f"👤 User watched: {', '.join(watch_history)}")
    print(f"\n🎯 Personalized Recommendations:\n")
    
    shown = 0
    for meta, dist in zip(recommendations['metadatas'][0], recommendations['distances'][0]):
        if meta['title'] in watch_history:
            continue
        print(f"  🌟 {meta['title']} ({meta['year']}) | {meta['genre']} | Match: {(1-dist)*100:.1f}%")
        shown += 1
        if shown >= n_results:
            break

# Simulate a user who likes action and psychological thrillers
profile_based_recommendations(["The Dark Knight", "Inception", "Mad Max: Fury Road"])

---
## Summary

| Concept | Key Takeaway |
|---------|-------------|
| Embeddings | Dense numerical representations that encode semantic meaning |
| Cosine Similarity | Measures angle between vectors; 1.0 = identical direction |
| ChromaDB | Open-source vector DB with HNSW index for efficient ANN search |
| Local Embeddings | sentence-transformers: free, fast, offline — ideal for development |
| Metadata Filtering | Combine semantic search with structured filters (year, genre) |
| Recommendations | Average user history embeddings → find nearest unwatch movies |

### Next Steps
- Module 4: Advanced RAG with FAISS — explore production-grade similarity search